In [1]:
import os
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from flax.training.train_state import TrainState
from IPython.display import FileLink, display

from flightning.algos import bptt
from flightning.envs import DynamicAvoidanceEnv
from flightning.modules import CNNLidarActor
from flightning.modules.observation_builder import ObservationBuilder
from flightning.visualization import RerunVizAdapter
from flightning.visualization.rerun_dynamic_avoidance import HAS_RERUN

print(os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"))
%matplotlib inline

None


# Training a Dynamic-Avoidance LiDAR Policy With BPTT and Rerun

## Seed it

In [2]:
seed = 0
key = jax.random.key(seed)
key_init, key_bptt, key_eval = jax.random.split(key, 3)

## Setup the Training Environment

`DynamicAvoidanceEnv` 的默认 `stop_lidar_grad=False` 对应本 change 的 `analytic_lidar_grad` 路线。

In [3]:
dt = 0.02

env = DynamicAvoidanceEnv(
    max_steps_in_episode=1000,
    dt=dt,
    delay=0.02,
    stop_lidar_grad=False,
)

obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
print(f"Observation dimension: {obs_dim}")
print(f"Action dimension: {action_dim}")

Observation dimension: 226
Action dimension: 4


## Define the Policy Network

In [4]:
# CNN encoder output: 9 * 3 * 16 = 432; state features: 10.
policy_net = CNNLidarActor(
    [442, 64, 64, action_dim],
    initial_scale=0.01,
    action_bias=env.hovering_action,
)
policy_params = policy_net.initialize(key_init)

## Setup the Optimizer and Train State

In [5]:
num_epochs = 1000
num_steps_per_epoch = 100
num_envs = 32

scheduler = optax.cosine_decay_schedule(1e-3, num_epochs)
tx = optax.adam(scheduler)
train_state = TrainState.create(
    apply_fn=policy_net.apply,
    params=policy_params,
    tx=tx,
)

## Train the Policy Using BPTT

In [ ]:
time_start = time.time()
res_dict = bptt.train(
    env,
    train_state,
    num_epochs=num_epochs,
    num_steps_per_epoch=num_steps_per_epoch,
    num_envs=num_envs,
    key=key_bptt,
    config=bptt.Config(logging=True, logging_freq=1),
)
time_train = time.time() - time_start
losses = jax.device_get(res_dict["metrics"])
print(f"Training time: {time_train:.2f}s")
print(f"Final loss: {float(losses[-1]):.4f}")

2026-06-11 14:54:26.343764: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-06-11 14:54:26.347447: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-06-11 14:54:26.347542: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-06-11 14:54:26.347578: W external/xla/xla/service/gpu/au

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 19700391512 bytes.

: 

In [ ]:
returns = -losses
plt.plot(returns)
plt.title(f"Final Return: {returns[-1]:.3f}, Training Time: {time_train:.1f}s")
plt.xlabel("Iteration")
plt.ylabel("Return")
plt.grid(True)
plt.show()

## Evaluate the Trained Policy and Export a Rerun Scene

In [ ]:
trained_state = res_dict["runner_state"].train_state

def policy_trained(obs, key):
    del key
    return trained_state.apply_fn(trained_state.params, obs)

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "flightning").exists():
    repo_root = repo_root.parent

rrd_path = repo_root / "examples" / "outputs" / "dynamic_avoidance_bptt_rerun.rrd"
rrd_path.parent.mkdir(parents=True, exist_ok=True)

viz = RerunVizAdapter(save_path=str(rrd_path))
state, _ = env.reset(key_eval)

# Default reset follows the P2M sector pattern and may begin outside the first-version
# termination box. For visualization, use a centered scene so the exported .rrd has
# a meaningful multi-step trajectory through the obstacle field.
quadrotor_state = state.quadrotor_state.replace(
    p=jnp.array([0.0, 0.0, 2.0]),
    R=jnp.eye(3),
    v=jnp.zeros(3),
    omega=jnp.zeros(3),
)
state = state.replace(
    quadrotor_state=quadrotor_state,
    start_pos=jnp.array([0.0, 0.0, 2.0]),
    target_pos=jnp.array([8.0, 0.0, 2.0]),
)
scan = env.lidar_sensor.get_scan(
    state.quadrotor_state.p,
    state.quadrotor_state.R,
    state.dobs_state.pos_xy,
    stop_lidar_grad=env.stop_lidar_grad,
)
obs = ObservationBuilder.get_observation(
    lidar_scan=scan,
    drone_pos=state.quadrotor_state.p,
    target_pos=state.target_pos,
    drone_vel=state.quadrotor_state.v,
    last_action=state.last_actions[-1],
)

total_reward = 0.0
logged_steps = 0
first_done_step = None
for step in range(120):
    key_eval, key_policy, key_step = jax.random.split(key_eval, 3)
    action = policy_trained(obs, key_policy)

    # env observation layout: lidar_flat(216) + target_dir(3) + velocity(3) + last_action(4).
    scan_image = obs[:216].reshape(1, 36, 6)
    viz.log_state(state, scan_image, step_idx=step)

    transition = env._step(state, action, key_step)
    state = transition.state
    obs = transition.obs
    total_reward += float(jax.device_get(transition.reward))
    logged_steps = step + 1

    done = bool(jax.device_get(transition.terminated | transition.truncated))
    if done and first_done_step is None:
        first_done_step = step
        break

print(f"Logged steps: {logged_steps}")
print(f"First done step: {first_done_step}")
print(f"Evaluation reward: {total_reward:.4f}")
if HAS_RERUN:
    print(f"RRD saved to: {rrd_path}")
    print(f"Open with: rerun {rrd_path}")
    display(FileLink(str(rrd_path)))
else:
    print("rerun-sdk is not installed; training ran, but .rrd export was skipped.")